In [15]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [16]:
import os, cv2, numpy as np, pandas as pd, tensorflow as tf

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, mean_absolute_error, r2_score

In [17]:
IMG_DIR = "/kaggle/input/datasets/amimulahasanrofik/abu-sayed/Fundus_CIMT_2903 Dataset"
CSV_PATH = "/kaggle/input/datasets/amimulahasanrofik/abu-csv/data_info.csv"

In [18]:
df = pd.read_csv(CSV_PATH)

df["image_name"] = df["image_name"].astype(str)
df["patient_id"] = df["image_name"].apply(lambda x: x.split("_")[0])

print("Total images:", len(df))
print("Unique patients:", df["patient_id"].nunique())
print(df["label"].value_counts())

Total images: 5806
Unique patients: 2903
label
1    4108
0    1698
Name: count, dtype: int64


In [19]:
print(os.listdir(IMG_DIR)[:10])

sample_path = os.path.join(IMG_DIR, df.iloc[0]["image_name"])
print(sample_path, os.path.exists(sample_path))

['499112002_R.png', '152584002_R.png', '425192002_R.png', '582061001_R.png', '250213001_R.png', '633289001_L.png', '598874001_R.png', '271971001_R.png', '451053004_R.png', '498562001_L.png']
/kaggle/input/datasets/amimulahasanrofik/abu-sayed/Fundus_CIMT_2903 Dataset/2491006_R.png True


In [20]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, val_idx in sgkf.split(df, df["label"], groups=df["patient_id"]):
    train_df = df.iloc[train_idx]
    val_df   = df.iloc[val_idx]
    break

print(len(train_df), len(val_df))

4644 1162


In [21]:
IMG_SIZE = 224

def crop_image(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    coords = np.column_stack(np.where(gray > 10))
    if len(coords) == 0:
        return img
    x,y,w,h = cv2.boundingRect(coords)
    return img[y:y+h, x:x+w]

def preprocess(path):
    img = cv2.imread(path)

    if img is None:
        return np.zeros((IMG_SIZE, IMG_SIZE, 3))  # safe fallback

    img = crop_image(img)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    # CLAHE
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l,a,b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0)
    cl = clahe.apply(l)
    img = cv2.merge((cl,a,b))
    img = cv2.cvtColor(img, cv2.COLOR_LAB2BGR)

    return img / 255.0

In [22]:
class DataGenerator(tf.keras.utils.Sequence):
    def __init__(self, df, img_dir, batch_size=16, augment=False, **kwargs):
        super().__init__(**kwargs)
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.batch_size = batch_size
        self.augment = augment

    def __len__(self):
        return len(self.df) // self.batch_size

    def __getitem__(self, idx):
        batch = self.df.iloc[idx*self.batch_size:(idx+1)*self.batch_size]

        imgs, cls, reg = [], [], []

        for _, row in batch.iterrows():
            path = os.path.join(self.img_dir, row["image_name"])

            img = preprocess(path)

            if self.augment:
                if np.random.rand() > 0.5:
                    img = np.fliplr(img)

            imgs.append(img)
            cls.append(row["label"])
            reg.append(row["thickness"])

        return np.array(imgs), {
            "cls": np.array(cls),
            "reg": np.array(reg)
        }

In [23]:
train_gen = DataGenerator(train_df, IMG_DIR, augment=True)
val_gen   = DataGenerator(val_df, IMG_DIR, augment=False)

In [24]:
base = tf.keras.applications.EfficientNetB4(
    include_top=False,
    input_shape=(224,224,3),
    weights="imagenet",
    pooling="avg"
)

x = base.output
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dense(256, activation="relu")(x)
x = tf.keras.layers.Dropout(0.4)(x)

# classification
cls = tf.keras.layers.Dense(128, activation="relu")(x)
cls_out = tf.keras.layers.Dense(1, activation="sigmoid", name="cls")(cls)

# regression
reg = tf.keras.layers.Dense(128, activation="relu")(x)
reg_out = tf.keras.layers.Dense(1, activation="linear", name="reg")(reg)

model = tf.keras.Model(inputs=base.input, outputs=[cls_out, reg_out])

In [25]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss={"cls":"binary_crossentropy","reg":"mse"},
    loss_weights={"cls":1.0,"reg":0.5},
    metrics={"cls":["accuracy"],"reg":["mae"]}
)

In [26]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=3),
]

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=30,
    callbacks=callbacks
)

Epoch 1/30


I0000 00:00:1776687444.512674     124 service.cc:152] XLA service 0x7e2054001530 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776687444.512715     124 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1776687456.473083     124 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-04-20 12:17:54.945546: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-20 12:17:55.128055: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-20 12:17:55.495956: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accur

290/290 ━━━━━━━━━━━━━━━━━━━━ 276s 471ms/step - cls_accuracy: 0.6160 - cls_loss: 0.6835 - loss: 1.0715 - reg_loss: 0.7760 - reg_mae: 0.6917 - val_cls_accuracy: 0.6797 - val_cls_loss: 0.6515 - val_loss: 0.8257 - val_reg_loss: 0.3484 - val_reg_mae: 0.5495 - learning_rate: 1.0000e-04
Epoch 2/30
290/290 ━━━━━━━━━━━━━━━━━━━━ 97s 335ms/step - cls_accuracy: 0.7431 - cls_loss: 0.5400 - loss: 0.7071 - reg_loss: 0.3342 - reg_mae: 0.4570 - val_cls_accuracy: 0.6927 - val_cls_loss: 0.6027 - val_loss: 0.6652 - val_reg_loss: 0.1249 - val_reg_mae: 0.2881 - learning_rate: 1.0000e-04
Epoch 3/30
290/290 ━━━━━━━━━━━━━━━━━━━━ 97s 334ms/step - cls_accuracy: 0.7489 - cls_loss: 0.5124 - loss: 0.6305 - reg_loss: 0.2362 - reg_mae: 0.3865 - val_cls_accuracy: 0.7083 - val_cls_loss: 0.5826 - val_loss: 0.6796 - val_reg_loss: 0.1941 - val_reg_mae: 0.3672 - learning_rate: 1.0000e-04
Epoch 4/30
290/290 ━━━━━━━━━━━━━━━━━━━━ 98s 338ms/step - cls_accuracy: 0.7687 - cls_loss: 0.4926 - loss: 0.5897 - reg_loss: 0.1943 - reg_

In [27]:
for layer in base.layers[-50:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss={"cls":"binary_crossentropy","reg":"mse"},
    loss_weights={"cls":1.0,"reg":0.5}
)

model.fit(train_gen, validation_data=val_gen, epochs=10)

Epoch 1/10
290/290 ━━━━━━━━━━━━━━━━━━━━ 232s 369ms/step - cls_accuracy: 0.8042 - cls_loss: 0.4235 - loss: 0.4969 - reg_loss: 0.1468 - reg_mae: 0.3024 - val_cls_accuracy: 0.7708 - val_cls_loss: 0.4925 - val_loss: 0.5542 - val_reg_loss: 0.1234 - val_reg_mae: 0.2903
Epoch 2/10
290/290 ━━━━━━━━━━━━━━━━━━━━ 96s 332ms/step - cls_accuracy: 0.8111 - cls_loss: 0.4154 - loss: 0.4812 - reg_loss: 0.1316 - reg_mae: 0.2871 - val_cls_accuracy: 0.7005 - val_cls_loss: 0.5830 - val_loss: 0.6350 - val_reg_loss: 0.1041 - val_reg_mae: 0.2632
Epoch 3/10
290/290 ━━━━━━━━━━━━━━━━━━━━ 99s 340ms/step - cls_accuracy: 0.8245 - cls_loss: 0.3952 - loss: 0.4588 - reg_loss: 0.1272 - reg_mae: 0.2854 - val_cls_accuracy: 0.7691 - val_cls_loss: 0.5326 - val_loss: 0.5827 - val_reg_loss: 0.1002 - val_reg_mae: 0.2595
Epoch 4/10
290/290 ━━━━━━━━━━━━━━━━━━━━ 98s 338ms/step - cls_accuracy: 0.8289 - cls_loss: 0.3759 - loss: 0.4359 - reg_loss: 0.1200 - reg_mae: 0.2767 - val_cls_accuracy: 0.7648 - val_cls_loss: 0.5286 - val_loss:

In [28]:
y_true_cls, y_pred_cls = [], []
y_true_reg, y_pred_reg = [], []

for X, y in val_gen:
    pred = model.predict(X, verbose=0)

    y_true_cls.extend(y["cls"])
    y_pred_cls.extend((pred[0] > 0.5).astype(int))

    y_true_reg.extend(y["reg"])
    y_pred_reg.extend(pred[1].flatten())

print(classification_report(y_true_cls, y_pred_cls))
print("MAE:", mean_absolute_error(y_true_reg, y_pred_reg))
print("R2:", r2_score(y_true_reg, y_pred_reg))

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


UnboundLocalError: cannot access local variable 'batch_outputs' where it is not associated with a value